In [92]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [93]:
import polars as pl
from datetime import datetime
from src.commons import data_loader

In [111]:
# Configuración profesional de visualización en Polars
pl.Config.set_tbl_rows(50)          # Muestra 10 filas máximo
pl.Config.set_fmt_str_lengths(300)   # Trunca textos largos a 15 caracteres para no romper la línea
pl.Config.set_tbl_width_chars(150)  # Ajusta el ancho total de la tabla en pantalla


polars.config.Config

## -- Constants Variables for File names

In [95]:
UTILITY_PROVIDER_FILE = "utility_provider.csv"
SUBSTATION_FILE = "substation.csv"
CONSUMER_FILE = "consumer.csv"
AMI_HEAD_END_FILE = "ami_head_end.csv"
DATA_MGMT_SYSTEM_FILE = "data_mgmt_system.csv"
DISTRIBUTION_NETWORK_FILE = "distribution_network.csv"
DISTRIBUTION_TRANSFORMER_FILE = "distribution_transformer.csv"
ENERGY_STORAGE_FILE = "energy_storage.csv"
POWER_PLANT_FILE = "power_plants.csv"
POWER_TRANSFORMER_FILE = "power_transformer.csv"
RENEWABLE_SOURCE_FILE = "renewable_source.csv"
SCADA_DMS_FILE = "scada_dms.csv"
SMART_METERS_FILE = "smart_meters.csv"

## -- Data Ingestion from CSV files

In [79]:
df_utility_provider = data_loader.read_csv(UTILITY_PROVIDER_FILE)

In [107]:
df_substation = data_loader.read_csv(SUBSTATION_FILE)

In [ ]:
df_consumer = data_loader.read_csv(CONSUMER_FILE)

In [ ]:
df_ami_head_end = data_loader.read_csv(AMI_HEAD_END_FILE)

In [ ]:
df_data_mgmt_system = data_loader.read_csv(DATA_MGMT_SYSTEM_FILE)

In [ ]:
df_distribution_network = data_loader.read_csv(DISTRIBUTION_NETWORK_FILE)

In [ ]:
df_distribution_transformer = data_loader.read_csv(DISTRIBUTION_TRANSFORMER_FILE)

In [ ]:
df_energy_storage = data_loader.read_csv(ENERGY_STORAGE_FILE)

In [ ]:
df_power_plants =  data_loader.read_csv(POWER_PLANT_FILE)

In [ ]:
df_power_transformer = data_loader.read_csv(POWER_TRANSFORMER_FILE)

In [ ]:
df_renewable_source = data_loader.read_csv(RENEWABLE_SOURCE_FILE)

In [ ]:
df_scada_dms = data_loader.read_csv(SCADA_DMS_FILE)

In [61]:
df_smart_meters = data_loader.read_csv(SMART_METERS_FILE)

## -- DATA CLEANSING & HARMONIZATION - UTILITY PROVIDER

In [100]:
# 1 Pipeline Execution
cleaned_utility_provider_df = (
    df_utility_provider
    # --- STEP 1: NAMING CONVENTIONS ---
    # Convert all columns to lowercase and replace spaces/hyphens with underscores
    .rename({col: col.lower().strip().replace(" ", "_") for col in df_utility_provider.columns})
    
  
    # --- STEP 2: DELETE EMPTY ROWS OR WHITE LINES ---
    # If ID of the substation is null, it is empty or white space, it removes the complete row.
    .filter(
        pl.col("provider_id").is_not_null() & 
        (pl.col("provider_id").str.strip_chars() != "")
    )
    # STEP 3: ADD INGESTION META DATA
    .with_columns([
        # Text alignment: trim whitespace and capitalize names cleanly
        pl.col("name").str.strip_chars(),
        pl.col("region").str.strip_chars(),
        # Adding ingestion metada
        pl.lit(datetime.now()).alias("ingested_at"), 
        pl.lit(UTILITY_PROVIDER_FILE).alias("ingested_from_file"),
    ])
    
    # --- STEP 4: TEXT HARMONIZATION ---
    # It will only apply "UNKNOWN" to empty cells actually empty from the valid rows
    .with_columns([
        pl.when(
            pl.col(col).is_null() | (pl.col(col).str.strip_chars() == "")
        )
        .then(pl.lit("UNKNOWN"))
        .otherwise(pl.col(col).str.strip_chars())
        .alias(col)
        for col in ["name", "region"]
    ])
    
    # --- STEP 5: DEDUPLICATION ---
    # Drop exact duplicates, keeping the first occurrence
    .unique(maintain_order=True)
    
    # Drop rows where critical identifying keys are missing completely
    .filter(pl.col("provider_id").is_not_null())
)

print(cleaned_utility_provider_df)


shape: (5, 5)
┌─────────────┬─────────────────────────┬─────────┬─────────────────┬──────────────────────┐
│ provider_id ┆ name                    ┆ region  ┆ ingested_at     ┆ ingested_from_file   │
│ ---         ┆ ---                     ┆ ---     ┆ ---             ┆ ---                  │
│ str         ┆ str                     ┆ str     ┆ datetime[μs]    ┆ str                  │
╞═════════════╪═════════════════════════╪═════════╪═════════════════╪══════════════════════╡
│ UP-001      ┆ Andes Power Co          ┆ North   ┆ 2026-08-28      ┆ utility_provider.csv │
│             ┆                         ┆         ┆ 04:44:58.346682 ┆                      │
│ UP-002      ┆ Pacifica Energy         ┆ South   ┆ 2026-08-28      ┆ utility_provider.csv │
│             ┆                         ┆         ┆ 04:44:58.346682 ┆                      │
│ UP-003      ┆ Northern Grid Utilities ┆ East    ┆ 2026-08-28      ┆ utility_provider.csv │
│             ┆                         ┆         ┆ 04:4

## -- DATA CLEANSING & HARMONIZATION   - SUBSTATION

In [112]:
# 1 Execution Pipeline
    
# 2. Pipeline cleansing data
cleaned_substation_df = (
    df_substation
    
    # --- STEP 1: NAMING CONVENSION ---
    .rename({col: col.strip().lower().replace(" ", "_") for col in df_substation.columns})
    
    # --- STEP 2: DROP EMPTY OR NULL ROWS ---
    # If the ID of the substation is null, or is empty, it dropss the complete row
    .filter(
        pl.col("substation_id").is_not_null() & 
        (pl.col("substation_id").str.strip_chars() != "")
    )
    
    # --- STEP 3: ADD INGESTION META DATA ---
    .with_columns([
        pl.col("voltage_kv").fill_null(0.0),
        pl.lit(datetime.now()).alias("ingested_at"), 
        pl.lit(SUBSTATION_FILE).alias("ingested_from_file"),
    ])
    
    # --- PASO 4: TEXT HARMONIZATION ---
    # Now it will only apply UNKNOWN" to empy valid rows
    .with_columns([
        pl.when(
            pl.col(col).is_null() | (pl.col(col).str.strip_chars() == "")
        )
        .then(pl.lit("UNKNOWN"))
        .otherwise(pl.col(col).str.strip_chars())
        .alias(col)
        for col in ["substation_type", "location", "source_type", "source_id", "scada_id"]
    ])
    
    # --- PASO 5: DEDUPLICATION ---
    .unique(maintain_order=True)
)

# 3. Show trust results
print(cleaned_substation_df)

shape: (500, 9)
┌───────────────┬─────────────────┬──────────────┬────────────┬───┬───────────┬──────────┬────────────────────────────┬────────────────────┐
│ substation_id ┆ substation_type ┆ location     ┆ voltage_kv ┆ … ┆ source_id ┆ scada_id ┆ ingested_at                ┆ ingested_from_file │
│ ---           ┆ ---             ┆ ---          ┆ ---        ┆   ┆ ---       ┆ ---      ┆ ---                        ┆ ---                │
│ str           ┆ str             ┆ str          ┆ f64        ┆   ┆ str       ┆ str      ┆ datetime[μs]               ┆ str                │
╞═══════════════╪═════════════════╪══════════════╪════════════╪═══╪═══════════╪══════════╪════════════════════════════╪════════════════════╡
│ SUB-0001      ┆ Step-down       ┆ District 80  ┆ 138.0      ┆ … ┆ PP-0004   ┆ SCD-003  ┆ 2026-08-28 04:47:54.246738 ┆ substation.csv     │
│ SUB-0002      ┆ Step-down       ┆ District 119 ┆ 138.0      ┆ … ┆ RS-0024   ┆ SCD-001  ┆ 2026-08-28 04:47:54.246738 ┆ substation.csv    

## -- DATA CLEANSING & HARMONIZATION   - CONSUMER

In [113]:
# 1 Execution Pipeline
    
# 2. Cleansing with polars
cleaned_consumer_df = (
    df_consumer
    
    # --- STEP 1: NAMING CONVENSION ---
    .rename({col: col.strip().lower().replace(" ", "_") for col in df_consumer.columns})
    
    # --- STEP 2: DROP EMPTY OR NULL ROWS---
    # if the ID of the consumer is null, or it is blank or white space, it drops the whole row
    .filter(
        pl.col("consumer_id").is_not_null() & 
        (pl.col("consumer_id").str.strip_chars() != "")
    )
    
    # --- STEP 3: INGESTION META DATA  ---
    .with_columns([
        pl.lit(datetime.now()).alias("ingested_at"), 
        pl.lit(SUBSTATION_FILE).alias("ingested_from_file"),
    ])
    
    # --- STEP 4: TEXT HARMONIZATION ---
    # Not it will only apply "UNKNOWN" to empty valid rows
    .with_columns([
        pl.when(
            pl.col(col).is_null() | (pl.col(col).str.strip_chars() == "")
        )
        .then(pl.lit("UNKNOWN"))
        .otherwise(pl.col(col).str.strip_chars())
        .alias(col)
        for col in ["account_type", "address"]
    ])
    
    # --- STEP 5: DEDUPLICATION ---
    .unique(maintain_order=True)
)

# 3. Show thrust result
print(cleaned_consumer_df)

NameError: name 'cleaned_consumer_df' is not defined